In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, accuracy_score
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
os.makedirs("../results", exist_ok=True)
label_names = ['Informative', 'Misinformative']

In [ ]:
with open("../config.json", "r") as f:
    all_configs = json.load(f)

args_keys = [key for key in all_configs.keys() if key.startswith("args")]
print(f"Evaluating {len(args_keys)} configurations")

In [ ]:
test_df = pd.read_csv("../results/test.csv")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

test_dataset = Dataset.from_pandas(test_df).map(
    lambda x: tokenizer(x["title"], padding="max_length", truncation=True, max_length=512), batched=True
).remove_columns(['title']).with_format('torch')

print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
def get_checkpoint_num(path):
    return int(os.path.basename(path).split("-")[-1])

all_model_results = []

for args_key in args_keys:
    print(f"\nEvaluating {args_key}...")
    config_args = all_configs[args_key]
    model_dir = config_args["output_dir"]
    
    checkpoints = sorted([os.path.join(model_dir, d) for d in os.listdir(model_dir) 
                          if d.startswith("checkpoint-")], 
                         key=get_checkpoint_num)
    
    all_checkpoint_results = []
    
    for checkpoint_path in checkpoints:
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
        trainer = Trainer(model=model)
        preds_output = trainer.predict(test_dataset)
        
        predictions = np.argmax(preds_output.predictions, axis=-1)
        true_labels = preds_output.label_ids
        
        all_checkpoint_results.append({
            "checkpoint": os.path.basename(checkpoint_path),
            "accuracy": float(accuracy_score(true_labels, predictions)),
            "f1": float(f1_score(true_labels, predictions, average='weighted')),
            "precision": float(precision_score(true_labels, predictions, average='weighted')),
        })
        
        del model, trainer
    
    best_result = max(all_checkpoint_results, key=lambda x: x["f1"])
    
    all_model_results.append({
        'config': args_key,
        'accuracy': best_result['accuracy'],
        'f1': best_result['f1'],
        'precision': best_result['precision']
    })
    
    results_file = os.path.join(model_dir, "test_results.json")
    with open(results_file, "w") as f:
        json.dump({"config": args_key, "checkpoints": all_checkpoint_results, "best": best_result}, f, indent=2)
    
    print(f"F1: {best_result['f1']:.4f}, Acc: {best_result['accuracy']:.4f}")

In [ ]:
if all_model_results:
    comparison_df = pd.DataFrame(all_model_results)
    comparison_df.to_csv('../results/test_comparison.csv', index=False)
    print("\n" + comparison_df.to_string(index=False))

In [ ]:
if all_model_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    metrics = ['accuracy', 'f1', 'precision']
    titles = ['Accuracy', 'F1 Score', 'Precision']
    
    for ax, metric, title in zip(axes, metrics, titles):
        ax.bar(comparison_df['config'], comparison_df[metric])
        ax.set_xlabel('Configuration')
        ax.set_ylabel(title)
        ax.set_title(f'{title} by Configuration')
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../results/test_metrics_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if all_model_results:
    best_model = max(all_model_results, key=lambda x: x['f1'])
    print("BEST MODEL ON TEST SET")
    print(f"Config: {best_model['config']}")
    print(f"F1: {best_model['f1']:.4f}, Acc: {best_model['accuracy']:.4f}, Precision: {best_model['precision']:.4f}")